# Predicting MS Conversion from a Clinically Isolated Syndrome
### The same four steps as the iris notebook — on real clinical data

Yesterday, with the **iris** flowers, we walked through the skeleton of almost every
machine-learning project:

1. **Preprocessing** — get the data into clean numbers
2. **Exploratory data analysis (EDA)** — look before you model
3. **Unsupervised learning** — does the data have structure on its own? (PCA, clustering)
4. **Classification** — can we predict the label? — and finally **validation**: do we *believe* the result?

The big idea of this course is that **those steps stay the same no matter the dataset.**
So today we keep the skeleton and swap in a real clinical problem.

**The question.** When someone has a first neurological episode — a *clinically isolated
syndrome (CIS)* — it sometimes turns out to be the first attack of *clinically definite
multiple sclerosis (CDMS)*, and sometimes it does not. Given what we measure at that first
visit (age, MRI lesions, spinal-fluid markers, …), **can we predict who will convert to MS?**

This mirrors a 2024 paper, *Interpretable Machine Learning for Predicting Multiple Sclerosis
Conversion from Clinically Isolated Syndrome* (Daniel et al.), and uses the same public data.

**Expect it to be messier than iris.** The iris species fell into tidy, separable clouds.
Patients will not. That gap — noisy features, missing values, few patients, imbalanced
classes — is not a failure of the method; **it is the lesson.**

## How to use this notebook

This notebook is built differently from a normal tutorial: **it does not give you the
answers.** Most code cells contain a `# TODO` with a description of what to do, and it is
your job to write the code and — more importantly — to *think about what comes out*.

You do not need to already know how to code:

- 💡 **Use Gemini (the AI built into Colab).** Click the Gemini / ✨ icon, or start a cell with
  the prompt box, and paste the `# TODO` description. Read the code it writes, run it, and if
  you get a red error message, paste the error back to Gemini and ask it to fix it.
- 🔧 **Turn the knobs.** Where a cell sets something like `feature = "Age (y)"`, change it and
  re-run. Poke at it. Break it. That is how the intuition forms.
- ✍️ **Answer the reflection questions out loud or in a text cell.** A plot you did not interpret
  taught you nothing. The questions in *italics* are the actual point of each section.

There are no wrong experiments here — only un-examined ones.

## 1. Setup

Install and import the tools. (On Google Colab these run as-is.)

In [ ]:
%pip install pandas numpy scikit-learn seaborn matplotlib openpyxl

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 2. Get the data

The data lives in a public GitHub repository with three files in its `Data` folder:

| File | What it is | We use it for |
|------|------------|---------------|
| `Mexican.xlsx` | 273 CIS patients, ~19 features, 10-year follow-up | the **main** analysis |
| `Lithuanian.xlsx` | 138 CIS patients, different hospital | **external validation** (section 8c) |
| `Concatenated Dataset.xlsx` | the shared features of both, merged | (optional, for stretch goals) |

Run the next cell to download it.

In [ ]:
!git clone https://github.com/tsantosh7/Multiple-Sclerosis-Conversion.git

import os
DATA = "Multiple-Sclerosis-Conversion/Data"
print(os.listdir(DATA))

### Loading the messy spreadsheet

Real data is rarely tidy. If you open `Mexican.xlsx`, the first few rows are blank or contain a
title, and the real column names sit on the **fourth** row. The `group` column codes the outcome
as **1 = converted to MS (CDMS)** and **2 = did *not* convert**, and a couple of rows are notes
rather than patients. The cell below handles all of that and builds one clean 0/1 outcome column
called `converted`. (This loading step is done for you — read the comments so you understand it.)

In [ ]:
# Column names are on row index 3 (the first three rows are blank / a title).
mex = pd.read_excel(f"{DATA}/Mexican.xlsx", header=3)

# Keep only real patients: rows whose 'group' is 1 or 2 (drops legend/empty rows).
mex = mex[pd.to_numeric(mex["group"], errors="coerce").isin([1, 2])].copy()

# Build a clean outcome: 1 = converted to MS, 0 = did not.
mex["converted"] = (pd.to_numeric(mex["group"]) == 1).astype(int)

print("Patients x columns:", mex.shape)
mex.head()

## 3. First look at the data

Before modelling anything, get to know the table. In iris this was quick (4 clean features,
3 balanced species). Here there are two new things to watch for that iris did not have:
**missing values** and **class imbalance**.

In [ ]:
# TODO: Inspect the data the way we did for iris.
#   - mex.shape           -> how many patients (rows) and features (columns)?
#   - mex.dtypes          -> which columns are numbers vs text?
#   - mex.describe()      -> ranges and averages of each feature
# Ask Gemini: "show shape, dtypes and summary statistics of a pandas dataframe called mex".

### Missing values and class balance

Two quick checks that decide a lot about what follows:

In [ ]:
# TODO (missingness): how many values are missing in each column?
#   Hint: mex.isna().sum()   (or mex.isna().mean() for a fraction)

# TODO (class balance): what fraction of patients actually converted?
#   Hint: mex["converted"].value_counts(normalize=True)

**Think about it (write your answers in a text cell):**

- *We have ~273 patients and ~19 features. In iris we had 150 samples and 4 features. Why does
  having **few patients but many features** make it easy for a model to "memorise" rather than
  learn? (This is the n-vs-p problem.)*
- *What fraction converted? If a lazy model just predicted "**nobody** converts" for every
  patient, what accuracy would it get? Keep that number in mind — we will compare to it later.*
- ✍️ **Data-dictionary task:** for each feature, write a one-line plain-English gloss of what it
  measures (e.g. *Oligoclonal bands = abnormal proteins in spinal fluid, a known MS marker*).
  Ask Gemini if a feature name is unfamiliar. You will need this when you interpret the model.

## 4. Preprocessing — and a trap to avoid

We need a feature matrix `X` and an outcome vector `y`. But not every column belongs in `X`.
One column in particular is a **trap**, and spotting it is one of the most important skills in
all of machine learning.

In [ ]:
# TODO: Decide which columns are PREDICTORS (go in X) and which must be left out.
#
#   - "Patient"  is just an ID number  -> not a predictor.
#   - "group" and "converted" are the OUTCOME -> NEVER put them in X.
#   - Look hard at " final EDSS". EDSS is a disability score. Ask yourself: is the
#     *final* one measured at the first visit, or years later at follow-up? If it is
#     only known at follow-up, can it honestly be used to PREDICT conversion?
#     (Using information you would not have at prediction time is called DATA LEAKAGE.)
#     Decide whether to drop it -- and write down your reasoning.
#
# feature_cols = [ ...choose them... ]
# X = mex[feature_cols]
# y = mex["converted"]

*Why does leakage matter? A model that secretly peeks at the future will look brilliant here
and then fail completely in a real clinic, where the future has not happened yet. Half of
"too-good-to-be-true" results are leakage.*

A note on encoding: most features here are already 0/1 or small numbers, so little encoding is
needed. One exception is `initial symptom`, whose values (1–10) are *categories*, not a scale —
"symptom 8" is not "twice symptom 4". For a first pass you can leave it as-is; as a stretch, try
one-hot encoding it and see if anything changes.

## 5. Exploratory data analysis

Same moves as iris — but now we colour by `converted` (did this patient go on to develop MS?)
instead of by flower species. We are looking for any feature, or pair of features, that pulls
the two groups apart.

### 5a. One feature at a time (univariate)

In [ ]:
feature = "Age (y)"   # <-- KNOB: try each column in turn

# TODO: draw a histogram of `feature` with converters and non-converters in different colours.
#   Hint: sns.histplot(data=mex, x=feature, hue="converted", ...).
#   Ask Gemini: "seaborn histogram of column `feature` in dataframe mex, split by column converted".

### 5b. Two features together (bivariate)

In [ ]:
# TODO: pick two features and plot one against the other, coloured by "converted".
#   For 0/1 features a scatter looks like a grid; a boxplot or sns.stripplot may show more.
#   Try a continuous feature (e.g. age) vs a binary one (e.g. an MRI lesion).

### 5c. Correlation between all features

In [ ]:
# TODO: compute and plot the correlation matrix of the numeric features.
#   Hint: mex[feature_cols].corr(), then sns.heatmap(..., annot=True, cmap="coolwarm").
#   Strong correlations hint that PCA (next section) can compress the features.

*After exploring: which **single** feature separates converters from non-converters best?
Is it a clean split like iris's petals, or do the two groups mostly overlap? What does that
overlap predict about how hard the classification task will be?*

## 6. Unsupervised learning: does it separate on its own?

Everything above used the outcome to colour the plots. **Unsupervised** methods ignore the
outcome and ask whether the patients fall into natural groups by themselves. We use **PCA**, exactly
as in iris: it builds new axes (principal components) that capture as much variation as possible
in as few dimensions as possible. **Standardise first**, so a feature does not dominate just
because of its units.

In [ ]:
# TODO: standardise the features, then fit PCA.
#   from sklearn.preprocessing import StandardScaler
#   from sklearn.decomposition import PCA
#   X_scaled = StandardScaler().fit_transform(X)     # (handle any NaNs first if needed)
#   pca = PCA().fit(X_scaled)
#   print pca.explained_variance_ratio_ and its cumulative sum (np.cumsum).

In [ ]:
# TODO: make a score plot — project the patients onto PC1 vs PC2 and colour by "converted".
#   scores = pca.transform(X_scaled)
#   Plot scores[:, 0] vs scores[:, 1], colour = y.

In [ ]:
# TODO: look at the loadings (pca.components_) for PC1 and PC2.
#   Which original features push hardest along PC1? What does "moving along PC1" mean clinically?

*The honest comparison with iris: there, two components held ~96% of the variance and the
species formed separate clouds. Here — how many components do you need to reach ~90%? Do
converters and non-converters separate in the PC1–PC2 plot, or do they sit on top of each other?
If they overlap, that is real clinical data telling you the truth: the signal is weak and
spread across many features.*

### 6b. Clustering — can we find the groups *without* the labels?

PCA gave us new axes; **clustering** does something different — it sorts each patient into a
**group** by similarity, never seeing the outcome. This is the same K-Means step you ran on the
iris flowers. The plan mirrors iris exactly:

1. cluster on the standardised features,
2. **plot** the clusters in the PCA plane (PCA is only for the picture),
3. cross-tabulate the clusters against who actually `converted`, to see whether the groups the
   algorithm found *on its own* line up with the disease.

We ask for **2** clusters because there are two outcomes. Run the prep cell, then fill in K-Means.

In [ ]:
# Prepared for you: a clean, standardised matrix to cluster on, plus 2D PCA coords for plotting.
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X_clust  = mex[feature_cols].apply(pd.to_numeric, errors="coerce")   # feature_cols from section 4
X_scaled = StandardScaler().fit_transform(
    SimpleImputer(strategy="median").fit_transform(X_clust))
pca_xy   = PCA(n_components=2).fit_transform(X_scaled)               # only for the 2D picture
y        = mex["converted"].values
print("ready to cluster:", X_scaled.shape)

In [ ]:
# TODO: run K-Means with 2 clusters on X_scaled, then judge it two ways.
#   from sklearn.cluster import KMeans
#   clusters = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X_scaled)
#
#   1) Cross-tabulate clusters against the true outcome to see if they line up:
#        pd.crosstab(clusters, y, rownames=["cluster"], colnames=["converted"])
#      Do the clusters mostly separate converters from non-converters, or are they mixed?
#
#   2) Make TWO scatter plots of pca_xy[:, 0] vs pca_xy[:, 1] side by side:
#      one coloured by the TRUE outcome y, one coloured by `clusters`.
#      Ask Gemini: "two seaborn scatter plots side by side coloured by different arrays".

### 6c. K-Means is just *one* kind of clustering — and not the best fit here

You probably saw the clusters only *weakly* match who converted. Part of that is the data (the
groups genuinely overlap) — but part is **K-Means itself**. Every clustering algorithm makes
**assumptions**, and K-Means makes ones this dataset breaks:

- It is **centroid-based**: it assumes clusters are roughly **round, equal-sized blobs** around a
  centre point, measured with straight-line (Euclidean) distance.
- Our features are mostly **0/1** (yes/no). "Round blobs in Euclidean space" is a poor description
  of yes/no data — there are no smooth clouds, just corners of a cube.

So the lesson is not "K-Means is bad", it is **match the algorithm to the shape of your data.**
The main families:

| Family | Examples | Good for / assumes | Watch out for |
|---|---|---|---|
| **Centroid** | K-Means, K-Medoids | Round, similar-sized blobs; you choose *k* | Poor on non-round or uneven clusters |
| **Distribution** | Gaussian Mixture (GMM) | Elliptical, overlapping blobs; gives *soft* probabilities | Still assumes Gaussian-ish shapes |
| **Hierarchical** | Agglomerative (+ dendrogram) | **Any** distance, incl. **Hamming/Jaccard for 0/1 data**; shows a tree | Slower; you still cut the tree to pick *k* |
| **Density** | DBSCAN, HDBSCAN | Arbitrary shapes; finds *k* itself; flags outliers as noise | Very sensitive to its `eps` setting |
| **Graph** | Spectral | Non-round, connected shapes | Heavier to compute |
| **Categorical** | **K-Modes / K-Prototypes** | Built for categorical / mixed data like ours | Separate package (`pip install kmodes`) |

The next cell runs several of these on the same data so you can compare them.

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score

methods = {
    "K-Means (centroid)":            KMeans(n_clusters=2, n_init=10, random_state=0),
    "Gaussian Mixture (elliptical)": GaussianMixture(n_components=2, random_state=0),
    "Agglomerative (hierarchical)":  AgglomerativeClustering(n_clusters=2),
    "Spectral (graph-based)":        SpectralClustering(n_clusters=2, affinity="nearest_neighbors", random_state=0),
    "DBSCAN (density, finds own k)": DBSCAN(eps=4.0, min_samples=5),
}

rows = []
for name, model in methods.items():
    labels  = model.fit_predict(X_scaled)
    n_found = len(set(labels) - {-1})              # DBSCAN marks outliers as -1
    shape   = silhouette_score(X_scaled, labels) if len(set(labels)) > 1 else float("nan")
    agree   = adjusted_rand_score(y, labels)        # 0 = random, 1 = perfect match to the outcome
    rows.append([name, n_found, round(shape, 3), round(agree, 3)])

pd.DataFrame(rows, columns=["method", "clusters found",
                            "silhouette (are clusters tight?)",
                            "ARI (do clusters match conversion?)"])

### What to look for when you cluster

Two different questions, two different kinds of score:

- **"Are the clusters tight and well-separated?"** — an *internal* score that needs no labels.
  **Silhouette** (−1 to 1, higher = tidier) is the usual one. Use it when you have no ground truth.
- **"Do the clusters match something I care about?"** — an *external* score comparing the clusters
  to a known label. **Adjusted Rand Index (ARI)**: **0 = no better than random, 1 = perfect**. We
  can only compute it here because we secretly *do* know who converted.

⚠️ **These two can disagree — and that is the most important thing to notice.** If you cluster the
raw 0/1 features with a **Hamming** distance (the proper choice for yes/no data), the silhouette
can look *better* (~0.48) while the ARI falls to ~0.00: the algorithm found tidy groups that have
**nothing to do with conversion**. A neat-looking cluster is not the same as a *useful* one.

**Choosing the number of clusters** (when the problem doesn't hand it to you): run k = 2, 3, 4… and
look for the "elbow" where more clusters stop helping, or pick the k with the best silhouette.
Density methods (DBSCAN) choose k for you — but only if you tune `eps` well.

*Reflection:*
- *In iris, K-Means recovered the flower species strongly (ARI well above 0.5). Here every method
  sits near ARI ≈ 0.1 — barely above random. What is the data telling you about whether "natural
  groups" and "who converts" are the same question?*
- *Which method made the **tightest** clusters (silhouette)? Was it the same one whose clusters best
  **matched conversion** (ARI)? If not, which score would actually matter to a doctor?*
- *DBSCAN probably lumped almost everyone into one cluster. Did it fail — or did it honestly report
  "I don't see well-separated density blobs here"?*

## 7. Classification — first, the *wrong* way (on purpose)

Now we use the labels and try to predict conversion. We will start by evaluating a model the
**wrong** way, because the mistake is so common and so seductive that you should feel it once,
deliberately, in a safe place.

In [ ]:
# TODO: fit a classifier on ALL the data, then score it on that SAME data.
#   from sklearn.linear_model import LogisticRegression
#   from sklearn.ensemble import RandomForestClassifier
#   - Standardise X (or use a Pipeline). Fit the model on (X, y).
#   - Print model.score(X, y)  -- its accuracy on the very data it learned from.
#   Try LogisticRegression first, then RandomForestClassifier.

*Write down the random forest's score. Now: would you trust a student who set their own exam,
took it with the answer key open, and then told you they got 100%? What did that number actually
measure? This is why the next section exists.*

## 8. Validation — do we actually believe the model?

This is the step you have been building toward, and the heart of the whole project. A model is
only as trustworthy as the way it was tested. We will test it three ways, each more honest than
the last.

### 8a. Internal validation: cross-validation

Instead of scoring on the training data, we **hold out** part of the data, train on the rest,
and score on the part the model never saw — then repeat so every patient gets a turn in the
held-out set. **Stratified** k-fold keeps the converter/non-converter ratio the same in each
fold.

In [ ]:
# TODO: evaluate honestly with stratified k-fold cross-validation.
#   from sklearn.model_selection import cross_val_score, StratifiedKFold
#   cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
#   scores = cross_val_score(model, X, y, cv=cv, scoring="balanced_accuracy")
#   Print scores.mean() and scores.std(), for LogisticRegression AND RandomForest.

*Compare three numbers now: the **naive** score (section 7), and the cross-validated scores for
**logistic regression** and the **random forest**. You may find the simpler model dropped to a
believable number while the random forest stayed suspiciously near-perfect — the Daniel et al.
paper likewise reported a random forest with a **perfect F1 of 1.0** on this very dataset of
~273 patients.*

*Here is the twist: if even cross-validation does **not** bring the random forest down to earth,
that is a clue, not a comfort. Ask yourself — could some features be near-copies of the diagnosis
itself? Several here (MRI lesions, oligoclonal bands) are literally part of how MS is **diagnosed**.
When the inputs almost define the label, a flexible model can look flawless and still have learned
nothing that would transfer to a new patient. Hold that suspicion — section 8c is where we find
out whether it was real skill or an illusion.*

### 8b. The right metrics for imbalanced data

Accuracy lies when classes are imbalanced (remember the "predict nobody converts" baseline from
section 3). So we look at the **confusion matrix** and metrics that cannot be fooled that way:
**sensitivity/recall** (of those who truly converted, how many did we catch?), **specificity**,
**balanced accuracy**, and **ROC-AUC**.

In [ ]:
# TODO: one honest train/test split, then look beyond accuracy.
#   from sklearn.model_selection import train_test_split
#   from sklearn.metrics import (confusion_matrix, classification_report,
#                                balanced_accuracy_score, roc_auc_score)
#   X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
#   Fit on train; predict on test. Print the confusion matrix, classification_report,
#   balanced_accuracy_score, and roc_auc_score (use predict_proba[:, 1] for AUC).
#   Tip: pass class_weight="balanced" to the model so it does not ignore the rarer class.

*Of the patients who really converted, what fraction did your model catch (recall for class 1)?
In a clinic, which mistake is worse here — telling a patient who will convert that they are fine
(a missed case), or worrying a patient who never converts (a false alarm)? Does your model make
the kind of mistake you can live with?*

### 8c. External validation: a brand-new hospital

Cross-validation still only ever saw **Mexican** patients. The real test of a model is whether it
works on data from a *different* population and a *different* hospital. We now train on the
Mexican cohort and test on the **Lithuanian** one.

The catch: the two hospitals recorded *different columns with different names*. Lining them up
into a set of **shared features** is itself part of the cost of external validation — so that
harmonisation is done for you below. There are also missing values in the Lithuanian data, which
your model will have to cope with.

In [ ]:
# Harmonisation: line up the 8 features both hospitals measured (done for you).
SHARED = ["Sex", "Age", "OCB", "VEP", "BAEP", "Periventricular", "Spinal", "Infratentorial"]

def load_mexican_shared():
    df = pd.read_excel(f"{DATA}/Mexican.xlsx", header=3)
    df = df[pd.to_numeric(df["group"], errors="coerce").isin([1, 2])].copy()
    out = pd.DataFrame({
        "Sex":             df["Gender"].map({1: 1, 2: 0}),
        "Age":             df["Age (y)"],
        "OCB":             df["Oligoclonal bands"],
        "VEP":             df["VEP"],
        "BAEP":            df["BAEP"],
        "Periventricular": df["Periventricular MRI"],
        "Spinal":          df["Spinal cord MRI"],
        "Infratentorial":  df["Infratentorial MRI"],
    }).apply(pd.to_numeric, errors="coerce")
    out["converted"] = (pd.to_numeric(df["group"]) == 1).astype(int).values
    return out

def load_lithuanian_shared():
    df = pd.read_excel(f"{DATA}/Lithuanian.xlsx", sheet_name="Database").dropna(subset=["MS"])
    out = pd.DataFrame({
        "Sex":             df["Sex"],
        "Age":             df["Age"],
        "OCB":             df["OGB + in CSF"],
        "VEP":             df["VEP +"],
        "BAEP":            df["BAEP +"],
        "Periventricular": df["Periventricular"],
        "Spinal":          df["MRI spinal lesions"],
        "Infratentorial":  df["MRI lesion localisation: infratentorally"],
    }).apply(pd.to_numeric, errors="coerce")
    out["converted"] = df["MS"].astype(int).values
    return out

mex_shared = load_mexican_shared()
lit_shared = load_lithuanian_shared()
print("Mexican   :", mex_shared.shape, "converted rate", round(mex_shared["converted"].mean(), 2))
print("Lithuanian:", lit_shared.shape, "converted rate", round(lit_shared["converted"].mean(), 2))

In [ ]:
# TODO: train on Mexican, test on Lithuanian.
#   X_train, y_train = mex_shared[SHARED], mex_shared["converted"]
#   X_test,  y_test  = lit_shared[SHARED], lit_shared["converted"]
#   Build a Pipeline that: (1) fills missing values  -> SimpleImputer
#                          (2) standardises           -> StandardScaler
#                          (3) classifies             -> e.g. RandomForest / LogisticRegression
#                                                        with class_weight="balanced"
#   Fit on the Mexican data ONLY. Predict on Lithuania. Print balanced accuracy and ROC-AUC.

*Compare this external-validation score with your cross-validation score from 8a. Which is
higher? Notice the two cohorts even converted at different rates (look at the printout above).
List three reasons a model might do worse on a new hospital's patients — different populations,
different equipment, different missing data. This is **domain shift**, and it is why a model that
shines in cross-validation can still fail in the real world. The paper's authors said the same:
an independent dataset is the real test — and now you have run it.*

- How did the random forest's score change across the three tests: **naive** (section 7) →
  **cross-validation** (8a) → **external** (8c)? Did cross-validation bring it down, or did the
  number only collapse at *external* validation? What does the timing of that collapse tell you?

## 10. Where this goes next

You have just reproduced the core of a published study. If you want to push further (this is the
arc of the full 7-day project):

- **Feature importance & interpretability** — which features drive the prediction? Compare random
  forest importances, *permutation importance*, and **SHAP**, then check them against the clinically
  known predictors (age, MRI lesion sites, oligoclonal bands).
- **Parsimony** — can a handful of features do almost as well as all of them?
- **Imbalance tools** — try SMOTE (`pip install imbalanced-learn`) to up-sample the rarer class,
  as the paper did, and see whether it helps honestly.
- **A neural network** — fit a small MLP and find out (probably) why deep learning loses to
  random forests on a few hundred rows of tabular data.

Same skeleton, every time: **preprocess → explore → reduce → classify → validate.**